In [9]:
# ============================================================================
# CARS.COM SCRAPER - VERSIONE LOCALE (PC)
# ============================================================================
# Logica: codice collega identico
# Setup: ChromeDriverManager per PC locale
# ============================================================================

import os
import json
import time
import random
import re
from typing import Dict, Any, List, Optional, Tuple
from datetime import datetime
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.common.exceptions import TimeoutException, WebDriverException
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

print("="*70)
print("CARS.COM SCRAPER - LOCAL VERSION")
print("="*70)

BASE_URL = "https://www.cars.com/research/"
REQUEST_COUNTER = 0



CARS.COM SCRAPER - LOCAL VERSION


In [10]:

# ============================================================================
# CONFIGURAZIONE PATHS LOCALI
# ============================================================================

# Modifica questi path per il tuo PC
BASE_DIR = r"./"
DATA_FOLDER = os.path.join(BASE_DIR, "data")
CHECKPOINT_FOLDER = os.path.join(BASE_DIR, "checkpoints")
LOGS_FOLDER = os.path.join(BASE_DIR, "logs")

# Crea cartelle se non esistono
for folder in [DATA_FOLDER, CHECKPOINT_FOLDER, LOGS_FOLDER]:
    os.makedirs(folder, exist_ok=True)


In [ ]:

# ============================================================================
# FUNZIONI 
# ============================================================================

def normalize_slug_part(s: pd.Series) -> pd.Series:
    return (
        s.astype(str)
         .str.strip()
         .str.lower()
         .str.replace(r"[\s\-]+", "_", regex=True)
         .str.replace(r"[^a-z0-9_]", "", regex=True)
         .str.replace(r"_+", "_", regex=True)
         .str.strip("_")
    )


def close_extra_tabs(driver: webdriver.Chrome) -> None:
    try:
        handles = driver.window_handles
        if len(handles) <= 1:
            return
        main = handles[0]
        for h in handles[1:]:
            driver.switch_to.window(h)
            driver.close()
        driver.switch_to.window(main)
    except Exception:
        pass


def polite_sleep(min_s: int = 30, max_s: int = 40, log_func=None) -> None:
    global REQUEST_COUNTER
    REQUEST_COUNTER += 1
    t = random.randint(min_s, max_s)
    msg = f"Request #{REQUEST_COUNTER} - sleep {t}s"
    if log_func:
        log_func(msg)
    else:
        print(msg)
    time.sleep(t)


def make_driver(headless: bool = True) -> webdriver.Chrome:
    options = Options()
    if headless:
        options.add_argument("--headless=new")

    options.add_argument("--window-size=1200,900")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--no-first-run")
    options.add_argument("--no-default-browser-check")
    options.add_argument("--disable-extensions")
    options.add_argument("--disable-popup-blocking")
    options.add_experimental_option("prefs", {
        "profile.default_content_setting_values.notifications": 2,
        "profile.default_content_setting_values.popups": 2,
    })
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)
    options.add_argument(
        "--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0 Safari/537.36"
    )
    
    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )
    driver.set_page_load_timeout(60)
    close_extra_tabs(driver)
    return driver


def safe_get(
    driver: webdriver.Chrome,
    url: str,
    headless: bool,
    wait_after_load: float = 1.0
) -> webdriver.Chrome:
    try:
        driver.get(url)
        close_extra_tabs(driver)
        time.sleep(wait_after_load)
        return driver
    except Exception as e:
        msg = str(e).lower()
        dead = (
            "connection refused" in msg
            or "invalid session id" in msg
            or "session deleted" in msg
            or "disconnected" in msg
            or isinstance(e, WebDriverException)
        )
        if not dead:
            raise

        try:
            driver.quit()
        except Exception:
            pass

        driver = make_driver(headless=headless)
        driver.get(url)
        close_extra_tabs(driver)
        time.sleep(wait_after_load)
        return driver


def build_research_query_url(marca_norm: str, modello_norm: str) -> str:
    return f"{BASE_URL}?make={marca_norm}&model={modello_norm}&year="


def build_reviews_url(marca_norm: str, modello_norm: str, year: int) -> str:
    return f"{BASE_URL}{marca_norm}-{modello_norm}-{year}/consumer-reviews/"


def get_years_from_dom_safe(
    driver: webdriver.Chrome,
    marca_norm: str,
    modello_norm: str,
    log_func,
    timeout: int = 12
) -> List[int]:
    url = build_research_query_url(marca_norm, modello_norm)
    log_func(f"GET {url}")
    driver.get(url)
    polite_sleep(log_func=log_func)

    wait = WebDriverWait(driver, timeout)

    try:
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "select#year-select")))
    except TimeoutException:
        return []

    try:
        wait.until(
            lambda d: any(
                (opt.get_attribute("value") or "").strip().isdigit()
                for opt in d.find_elements(By.CSS_SELECTOR, "select#year-select option")
            )
        )
    except TimeoutException:
        return []

    years: List[int] = []
    for opt in driver.find_elements(By.CSS_SELECTOR, "select#year-select option"):
        v = (opt.get_attribute("value") or "").strip()
        if v.isdigit() and len(v) == 4:
            years.append(int(v))

    return sorted(set(years), reverse=True)


def get_reviews_count_robust(driver: webdriver.Chrome, timeout: int = 18) -> int:
    nums: List[int] = []
    wait = WebDriverWait(driver, timeout)

    try:
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, ".sds-rating__link")))
    except TimeoutException:
        pass

    try:
        for el in driver.find_elements(By.CSS_SELECTOR, ".sds-rating__link"):
            txt = (el.text or "").lower()
            if "review" not in txt:
                continue
            m = re.search(r"\d+", txt)
            if m:
                nums.append(int(m.group()))
    except Exception:
        pass

    try:
        body = (driver.find_element(By.TAG_NAME, "body").text or "").lower()
        for m in re.finditer(r"(\d+)\s+review", body):
            nums.append(int(m.group(1)))
    except Exception:
        pass

    return max(nums) if nums else 0


def get_rating_breakdown(driver: webdriver.Chrome, wait_seconds: int = 15) -> Dict[str, float]:
    breakdown: Dict[str, float] = {}
    wait = WebDriverWait(driver, wait_seconds)

    try:
        items = wait.until(
            EC.presence_of_all_elements_located((By.CSS_SELECTOR, "ul.review-breakdown--list li"))
        )
    except TimeoutException:
        return {}

    for item in items:
        try:
            name = item.find_element(By.CSS_SELECTOR, "span.sds-definition-list__display-name").text.strip()
            value = item.find_element(By.CSS_SELECTOR, "span.sds-definition-list__value").text.strip()
            breakdown[name] = float(value)
        except Exception:
            continue

    return breakdown


def weighted_breakdown_average(per_year: List[Dict[str, Any]]) -> Dict[str, float]:
    sums: Dict[str, float] = {}
    wsum: Dict[str, float] = {}

    for rec in per_year:
        w = int(rec.get("reviews", 0) or 0)
        bd = rec.get("rating_breakdown") or {}
        if w <= 0 or not bd:
            continue
        for k, v in bd.items():
            try:
                fv = float(v)
            except Exception:
                continue
            sums[k] = sums.get(k, 0.0) + fv * w
            wsum[k] = wsum.get(k, 0.0) + w

    out: Dict[str, float] = {}
    for k in sums:
        if wsum.get(k, 0.0) > 0:
            out[k] = round(sums[k] / wsum[k], 4)
    return out


def save_checkpoint(cars: List[Dict[str, Any]], checkpoint_path: str) -> None:
    os.makedirs(os.path.dirname(checkpoint_path) or ".", exist_ok=True)
    tmp_path = checkpoint_path + ".tmp"
    with open(tmp_path, "w", encoding="utf-8") as f:
        json.dump(cars, f, indent=2, ensure_ascii=False)
    os.replace(tmp_path, checkpoint_path)


def load_checkpoint_if_exists(checkpoint_path: str) -> Optional[List[Dict[str, Any]]]:
    if os.path.exists(checkpoint_path):
        with open(checkpoint_path, "r", encoding="utf-8") as f:
            return json.load(f)
    return None


def process_one_car(
    driver: webdriver.Chrome,
    car: Dict[str, Any],
    headless: bool,
    threshold: int,
    sleep_min: int,
    sleep_max: int,
    log_func,
) -> Tuple[webdriver.Chrome, Dict[str, Any]]:
    mk = car["marca_norm"]
    md = car["modello_norm"]

    years = get_years_from_dom_safe(driver, mk, md, log_func, timeout=12)

    if not years:
        car["done"] = True
        car["not_found"] = True
        car["error"] = "not_found_on_carscom"
        car["per_year"] = []
        car["rating_breakdown_avg_weighted"] = {}
        car["review_cumulata"] = 0
        car["stop_year"] = None
        log_func("NOT FOUND on cars.com")
        return driver, car

    cumulata = 0
    per_year: List[Dict[str, Any]] = []

    log_func(f"Max year = {years[0]}")
    log_func(f"Available years (desc): {years}")
    log_func("Cumulative reviews = 0")

    for y in years:
        url = build_reviews_url(mk, md, y)
        log_func(f"GET {url}")
        driver = safe_get(driver, url, headless=headless, wait_after_load=1.0)
        polite_sleep(sleep_min, sleep_max, log_func=log_func)

        reviews = get_reviews_count_robust(driver)
        bd = get_rating_breakdown(driver, wait_seconds=15)

        cumulata += reviews
        per_year.append({
            "anno": y,
            "url": url,
            "reviews": reviews,
            "cumulata": cumulata,
            "rating_breakdown": bd
        })

        log_func(f"  {y}: +{reviews} = {cumulata} | breakdown={list(bd.keys())}")

        if cumulata >= threshold:
            break

    car["per_year"] = per_year
    car["review_cumulata"] = cumulata
    car["stop_year"] = per_year[-1]["anno"] if per_year else None
    car["rating_breakdown_avg_weighted"] = weighted_breakdown_average(per_year)
    car["done"] = True
    car["not_found"] = False
    car.pop("error", None)

    return driver, car


def build_flat_db_weighted(cars: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    flat: List[Dict[str, Any]] = []
    for car in cars:
        avg = car.get("rating_breakdown_avg_weighted", {}) or {}
        flat.append({
            "marca_norm": car.get("marca_norm"),
            "modello_norm": car.get("modello_norm"),
            "found_on_carscom": not bool(car.get("not_found", False)),
            "cumulata": car.get("review_cumulata", 0),
            "comfort": avg.get("Comfort"),
            "interior": avg.get("Interior"),
            "performance": avg.get("Performance"),
            "value": avg.get("Value"),
            "exterior": avg.get("Exterior"),
            "reliability": avg.get("Reliability"),
        })
    return flat



In [12]:

# ============================================================================
# MAIN FUNCTION
# ============================================================================

def scrape_cars_reviews(
    autoscout_json_path: str,
    headless: bool = True,
    threshold: int = 50,
    sleep_min: int = 30,
    sleep_max: int = 40,
    retries_per_car: int = 2,
    restart_driver_every: int = 30,
    resume_if_done: bool = True,
    max_vehicles: Optional[int] = None
):
    
    # Setup paths
    checkpoint_path = os.path.join(CHECKPOINT_FOLDER, 'carscom_checkpoint.json')
    output_enriched_path = os.path.join(DATA_FOLDER, 'carscom_enriched.json')
    output_flat_path = os.path.join(DATA_FOLDER, 'carscom_flat_weighted.json')
    log_file = os.path.join(LOGS_FOLDER, f'carscom_log_{datetime.now().strftime("%Y%m%d_%H%M%S")}.txt')
    
    # Logging
    def log(message):
        timestamp = datetime.now().strftime("%H:%M:%S")
        log_message = f"[{timestamp}] {message}"
        print(log_message)
        with open(log_file, 'a', encoding='utf-8') as f:
            f.write(log_message + '\n')
    
    log("="*70)
    log("CARS.COM SCRAPING - START")
    log("="*70)
    log(f"Autoscout input: {autoscout_json_path}")
    log(f"Checkpoint: {checkpoint_path}")
    log(f"Output enriched: {output_enriched_path}")
    log(f"Output flat: {output_flat_path}")
    log(f"Log file: {log_file}")
    log(f"Threshold: {threshold} reviews")
    log("="*70)
    
    # Load autoscout data
    with open(autoscout_json_path, "r", encoding="utf-8") as f:
        autoscout = json.load(f)
    
    combined = autoscout.get("combined_data", [])
    if not combined:
        log("[ERROR] combined_data vuoto")
        return
    
    log(f"Loaded {len(combined)} cars from autoscout")
    
    # Extract unique marca/modello
    df = pd.DataFrame(combined)
    if "make" not in df.columns or "model" not in df.columns:
        log("[ERROR] Mancano campi 'make'/'model'")
        return
    
    unique_mm = df[["make", "model"]].dropna().drop_duplicates().reset_index(drop=True)
    unique_mm["marca_norm"] = normalize_slug_part(unique_mm["make"])
    unique_mm["modello_norm"] = normalize_slug_part(unique_mm["model"])
    
    base_cars: List[Dict[str, Any]] = unique_mm.to_dict(orient="records")
    for c in base_cars:
        c["marca"] = c["make"]
        c["modello"] = c["model"]
        c.pop("make", None)
        c.pop("model", None)
        c.setdefault("done", False)
        c.setdefault("not_found", False)
        c.setdefault("per_year", [])
        c.setdefault("rating_breakdown_avg_weighted", {})
        c.setdefault("review_cumulata", 0)
        c.setdefault("stop_year", None)
    
    log(f"Unique marca/modello: {len(base_cars)}")
    
    # Limit per test
    if max_vehicles:
        base_cars = base_cars[:max_vehicles]
        log(f"Limited to {max_vehicles} vehicles for testing")
    
    # Check checkpoint
    checkpoint_data = load_checkpoint_if_exists(checkpoint_path)
    if checkpoint_data and resume_if_done:
        log(f"\n[CHECKPOINT] Found previous session")
        log(f"             Vehicles in checkpoint: {len(checkpoint_data)}")
        completed = sum(1 for c in checkpoint_data if c.get('done'))
        log(f"             Completed: {completed}/{len(checkpoint_data)}")
        
        user_input = input("\nResume from checkpoint? (y/n): ").lower()
        if user_input == 'y':
            base_cars = checkpoint_data
            log("Resuming from checkpoint\n")
        else:
            log("Starting fresh\n")
    
    total = len(base_cars)
    driver = make_driver(headless=headless)
    start_time = time.time()
    
    try:
        for idx, car in enumerate(base_cars, start=1):
            
            # RESUME
            if resume_if_done and car.get("done") is True:
                log(f"\n[{idx}/{total}] SKIP (done): {car['marca_norm']}/{car['modello_norm']}")
                continue
            
            # RESTART DRIVER
            if restart_driver_every > 0 and idx > 1 and (idx - 1) % restart_driver_every == 0:
                log(f"\nRestarting driver (every {restart_driver_every} vehicles)...")
                try:
                    driver.quit()
                except Exception:
                    pass
                driver = make_driver(headless=headless)
            
            log(f"\n{'='*70}")
            log(f"VEHICLE {idx}/{total}")
            log(f"Marca: {car['marca']} -> {car['marca_norm']}")
            log(f"Modello: {car['modello']} -> {car['modello_norm']}")
            log(f"{'='*70}")
            
            last_err = None
            
            for attempt in range(1, retries_per_car + 1):
                try:
                    log(f"      > Attempt {attempt}/{retries_per_car}")
                    driver, car = process_one_car(
                        driver, car, headless, threshold, sleep_min, sleep_max, log
                    )
                    last_err = None
                    break
                
                except WebDriverException as e:
                    last_err = e
                    log(f"      WebDriver error (attempt {attempt}/{retries_per_car}): {e}")
                    try:
                        driver.quit()
                    except Exception:
                        pass
                    driver = make_driver(headless=headless)
                    time.sleep(random.uniform(2, 5))
                
                except Exception as e:
                    last_err = e
                    log(f"      Error (attempt {attempt}/{retries_per_car}): {e}")
                    time.sleep(random.uniform(2, 5))
            
            if last_err is not None:
                car["done"] = True
                car["error"] = str(last_err)
                car["per_year"] = []
                car["rating_breakdown_avg_weighted"] = {}
                car["review_cumulata"] = 0
                log(f"      > Status: FAILED after {retries_per_car} attempts")
            
            # CHECKPOINT
            save_checkpoint(base_cars, checkpoint_path)
            
            # Progress
            elapsed = time.time() - start_time
            avg_time = elapsed / idx
            remaining = avg_time * (total - idx)
            completed = sum(1 for c in base_cars if c.get('done'))
            
            log(f"      [Checkpoint Saved]")
            log(f"      Progress: {completed}/{total} ({completed/total*100:.1f}%)")
            log(f"      Elapsed: {elapsed/60:.1f} min | ETA: {remaining/60:.1f} min")
    
    finally:
        try:
            driver.quit()
        except Exception:
            pass
    
    # Save final outputs
    log(f"\n{'='*70}")
    log("SAVING FINAL OUTPUTS")
    log(f"{'='*70}")
    
    with open(output_enriched_path, 'w', encoding='utf-8') as f:
        json.dump(base_cars, f, indent=2, ensure_ascii=False)
    log(f"Enriched data saved: {output_enriched_path}")
    
    flat_db = build_flat_db_weighted(base_cars)
    with open(output_flat_path, 'w', encoding='utf-8') as f:
        json.dump(flat_db, f, indent=2, ensure_ascii=False)
    log(f"Flat weighted DB saved: {output_flat_path}")
    
    # Cleanup checkpoint
    if os.path.exists(checkpoint_path):
        os.remove(checkpoint_path)
        log(f"Checkpoint removed (task complete)")
    
    # Stats
    found = sum(1 for c in base_cars if not c.get('not_found'))
    not_found = sum(1 for c in base_cars if c.get('not_found'))
    total_reviews = sum(c.get('review_cumulata', 0) for c in base_cars)
    
    log(f"\n{'='*70}")
    log("FINAL STATISTICS")
    log(f"{'='*70}")
    log(f"Total vehicles: {total}")
    log(f"Found on cars.com: {found}")
    log(f"Not found: {not_found}")
    log(f"Total reviews cumulated: {total_reviews}")
    log(f"\n{'='*70}")
    log("SCRAPING COMPLETED")
    log(f"{'='*70}\n")
    
    print(f"\n[SUCCESS] Scraping completed!")
    print(f"Results saved to:")
    print(f"  - Enriched: {output_enriched_path}")
    print(f"  - Flat DB: {output_flat_path}")
    print(f"  - Log: {log_file}")



In [13]:

# ============================================================================
# ESECUZIONE
# ============================================================================

if __name__ == "__main__":
    # MODIFICA QUESTO PATH
    autoscout_path = os.path.join(DATA_FOLDER, "autoscout24_combined_data.json")
    
    # Test con 5 veicoli
    scrape_cars_reviews(
    autoscout_json_path=autoscout_path,
    headless=True,
    threshold=50,
    sleep_min=5,  # da 30 a 15
    sleep_max=10,  # da 40 a 25
   )

[00:38:19] ======================================================================
[00:38:19] CARS.COM SCRAPING - START
[00:38:19] ======================================================================
[00:38:19] Autoscout input: ./data/autoscout24_combined_data.json
[00:38:19] Checkpoint: ./checkpoints/carscom_checkpoint.json
[00:38:19] Output enriched: ./data/carscom_enriched.json
[00:38:19] Output flat: ./data/carscom_flat_weighted.json
[00:38:19] Log file: ./logs/carscom_log_20251230_003819.txt
[00:38:19] Threshold: 50 reviews
[00:38:19] ======================================================================
[00:38:19] Loaded 3532 cars from autoscout
[00:38:20] Unique marca/modello: 705
[00:38:22] 
[00:38:22] VEHICLE 1/705
[00:38:22] Marca: bmw -> bmw
[00:38:22] Modello: 320 -> 320
[00:38:22] ======================================================================
[00:38:22]       > Attempt 1/2
[00:38:22] GET https://www.cars.com/research/?make=bmw&model=320&year=
[00:38:23] Request #1